[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/adaptive-data/process_unstructured_documents.ipynb)

# Processing Unstructured Documents

Turn a document (PDF here, but DOCX / PPTX / XLSX / HTML work too) into dataset rows with the API: upload, quote extraction, split into rows, then adapt.

**Docs:** [Processing unstructured documents](https://docs.adaptionlabs.ai/runbooks/processing-unstructured-documents/) · [Create a dataset](https://docs.adaptionlabs.ai/adaptive-data/create-a-dataset/#upload-documents)  
**Prerequisites:** an Adaption API key ([create one](https://adaptionlabs.ai/app/settings?tab=api_keys)) set as `ADAPTION_API_KEY` (env var or Colab secret).

> ⏱ Runtime: minutes · 💳 Cost: low — page extraction plus a small adapt run

In [ ]:
%pip install -q adaption fpdf2 pandas pyarrow

## Setup

Load your API key. In Colab, add `ADAPTION_API_KEY` to the secrets panel (🔑 in the sidebar) and enable it for this notebook. Running locally, the cell prompts for the key if the environment variable is not set.

In [ ]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

## Parameters

In [ ]:
# How extracted content becomes rows:
#   "per_page"     -> one row per page (more training examples)
#   "per_document" -> one row per file (long-context / whole-document tasks)
PREPROCESSING_MODE = "per_page"

# Keep the adaptation run small while you iterate.
MAX_ROWS = 50

# After extraction the page/document text lives in one column. The app names it
# for you; confirm it under Datasets -> your dataset -> Columns in the Adaption
# app, then set it here before Step 5.
CONTENT_COLUMN = None

## Step 1 — Create a sample document

Normally you would bring your own file. Here we generate a short multi-page PDF so the notebook is self-contained. Supported document types include `.pdf`, `.docx`, `.pptx`, `.xlsx`, `.html`, `.txt`, and `.zip`.

In [ ]:
from fpdf import FPDF

pages = [
    (
        "Returns policy",
        "Customers may return most electronics within 30 days of delivery for a full "
        "refund. Items must be in their original packaging with all accessories. "
        "Opened software and gift cards are non-refundable.",
    ),
    (
        "Shipping policy",
        "Standard shipping takes three to five business days. Expedited shipping "
        "arrives in two business days. Orders over 50 dollars ship free within the "
        "continental United States.",
    ),
    (
        "Warranty",
        "All devices include a 12-month limited warranty covering manufacturing "
        "defects. The warranty does not cover accidental damage. Customers can start "
        "a claim from their account dashboard.",
    ),
]

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
for title, body in pages:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.multi_cell(0, 10, title)
    pdf.ln(4)
    pdf.set_font("Helvetica", size=12)
    pdf.multi_cell(0, 8, body)

pdf.output("company_handbook.pdf")
print("Wrote company_handbook.pdf")

## Step 2 — Upload the document

Documents are not rows yet. After the upload, Adaption counts pages and the dataset stops at `awaiting_input` with **no** `row_count`, waiting for you to launch extraction.

In [ ]:
dataset = client.datasets.upload_file("company_handbook.pdf")
dataset_id = dataset.dataset_id

status = client.datasets.wait_for_completion(dataset_id)
print("Status:", status.status, "| row_count:", status.row_count)

## Step 3 — Quote the extraction cost

Page counting runs in the background after upload. `get_preprocessing_credits` returns the page count and the credits extraction will reserve.

In [ ]:
quote = client.datasets.get_preprocessing_credits(dataset_id)
print(f"{quote.total_pages:.0f} pages cost {quote.extraction_cost} credits")
print(f"{quote.available_credits} credits available")
if quote.insufficient_credits:
    raise RuntimeError("Not enough credits to extract these documents")

## Step 4 — Launch extraction

`launch_preprocessing` reserves credits and starts turning the document into rows. Launch once per dataset — a second call returns HTTP 409. When extraction finishes, the dataset returns to `awaiting_input`, this time with a `row_count`.

In [ ]:
launch = client.datasets.launch_preprocessing(
    dataset_id,
    preprocessing_mode=PREPROCESSING_MODE,
)
print(f"Reserved {launch.credits_escrowed} credits")

status = client.datasets.wait_for_completion(dataset_id)
if status.status == "failed":
    err = status.error_data
    msg = (err and err.message) or "unknown error"
    raise RuntimeError(f"Document extraction failed: {msg}")
print(f"Extracted {status.row_count} rows")

## Step 5 — Map the extracted content

Extraction produced regular rows with the text in one column. Two common mappings:

- **Per page → completion.** Treat each page as the answer and let Adaptive Data write a matching prompt.
- **Per document → context + universal prompt.** Apply one instruction to every document, with the document supplied as context.

Set `CONTENT_COLUMN` in the parameters cell to the extracted text column (confirm the name in the app under **Datasets → your dataset → Columns**). The cell stops loudly until you do.

In [ ]:
if CONTENT_COLUMN is None:
    raise ValueError(
        "Set CONTENT_COLUMN in the parameters cell to the extracted text column. "
        "Find its name in the Adaption app: Datasets -> your dataset -> Columns."
    )

if PREPROCESSING_MODE == "per_page":
    column_mapping = {"completion": CONTENT_COLUMN}
else:
    column_mapping = {
        "universal_prompt": (
            "Summarize this document in three sentences, then list any dates, "
            "amounts, and policies it mentions."
        ),
        "context": [CONTENT_COLUMN],
    }

print(column_mapping)

## Step 6 — Adapt and download

From here it is a normal Adaptive Data run. Estimate first, then adapt and download the result.

In [ ]:
estimate = client.datasets.adapt(
    dataset_id,
    column_mapping=column_mapping,
    job_specification={"max_rows": MAX_ROWS},
    estimate=True,
)
print(f"Estimated credits: {estimate.estimated_credits_consumed}")

In [ ]:
client.datasets.adapt(
    dataset_id,
    column_mapping=column_mapping,
    job_specification={"max_rows": MAX_ROWS},
)
status = client.datasets.wait_for_completion(dataset_id, timeout=3600)
if status.status == "failed":
    raise RuntimeError(f"Adaptation failed: {status.error_data}")

client.datasets.download(dataset_id, file_format="csv").write_to_file(
    "documents_adapted.csv"
)
print("Wrote documents_adapted.csv")

In [ ]:
import pandas as pd

pd.read_csv("documents_adapted.csv").head()

## Next steps

- [Configure Adaptive Data](https://docs.adaptionlabs.ai/adaptive-data/configure-adaptive-data/) to add recipes and brand controls.
- [Evaluate dataset quality](https://docs.adaptionlabs.ai/adaptive-data/evaluate-dataset-quality/) on the result.

## Clean up

In [ ]:
# client.datasets.delete(dataset_id=dataset_id)